In [1]:
import pandas as pd
import numpy as np

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Clustering
from sklearn.cluster import KMeans

# Visualization
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("BSG.csv")

# Preview data
df.head()

,created_utc,score,domain,id,title,author,ups,downs,num_comments,permalink,...,over_18,thumbnail,subreddit_id,edited,link_flair_css_class,author_flair_css_class,is_self,name,url,distinguished
0,1.362692e+09,911,i.imgur.com,19vc1i,How I feel looking at the front page of reddit,chiachia2,1013,102,34,http://www.reddit.com/r/BSG/comments/19vc1i/ho...,...,False,http://b.thumbs.redditmedia.com/EKoL_Gdx01meou...,t5_2qlvu,False,NaN,NaN,False,t3_19vc1i,http://i.imgur.com/MeMbQDP.jpg,NaN
1,1.360106e+09,580,imgur.com,17yoix,"Me as Starbuck, with President Roslin and Admi...",_galacticat,628,48,49,http://www.reddit.com/r/BSG/comments/17yoix/me...,...,False,http://f.thumbs.redditmedia.com/Nf3jxd-5AdZ9DB...,t5_2qlvu,False,NaN,NaN,False,t3_17yoix,http://imgur.com/a/2NT47,NaN
2,1.354915e+09,483,imgur.com,14gpws,Just Another Reason Why BSG Makes Sense,Pharazlyg,548,65,60,http://www.reddit.com/r/BSG/comments/14gpws/ju...,...,False,http://b.thumbs.redditmedia.com/q-H_p3BNmVMfg-...,t5_2qlvu,False,NaN,NaN,False,t3_14gpws,http://imgur.com/o9q0i,NaN
3,1.369706e+09,466,i.imgur.com,1f69k2,A homemade bleached shirt I made: Launch alert...,Ferenginar,514,48,21,http://www.reddit.com/r/BSG/comments/1f69k2/a_...,...,False,http://b.thumbs.redditmedia.com/UDRY7Sfuu0hFnI...,t5_2qlvu,False,NaN,NaN,False,t3_1f69k2,http://i.imgur.com/PIXWz5e.jpg,NaN
4,1.367958e+09,461,i.imgur.com,1dvts8,"The owner saw me taking a picture, said ""Watch...",dishie,507,46,9,http://www.reddit.com/r/BSG/comments/1dvts8/th...,...,False,http://d.thumbs.redditmedia.com/jCJ0VgGRGOscDJ...,t5_2qlvu,False,NaN,NaN,False,t3_1dvts8,http://i.imgur.com/pNoCWUe.jpg,NaN


In [3]:
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   created_utc             1000 non-null   float64
 1   score                   1000 non-null   int64  
 2   domain                  1000 non-null   str    
 3   id                      1000 non-null   str    
 4   title                   1000 non-null   str    
 5   author                  969 non-null    str    
 6   ups                     1000 non-null   int64  
 7   downs                   1000 non-null   int64  
 8   num_comments            1000 non-null   int64  
 9   permalink               1000 non-null   str    
 10  selftext                141 non-null    str    
 11  link_flair_text         0 non-null      float64
 12  over_18                 1000 non-null   bool   
 13  thumbnail               1000 non-null   str    
 14  subreddit_id            1000 non-null   str    
 15 

Index(['created_utc', 'score', 'domain', 'id', 'title', 'author', 'ups',
       'downs', 'num_comments', 'permalink', 'selftext', 'link_flair_text',
       'over_18', 'thumbnail', 'subreddit_id', 'edited',
       'link_flair_css_class', 'author_flair_css_class', 'is_self', 'name',
       'url', 'distinguished'],
      dtype='str')

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
print(df.columns.tolist())

['created_utc', 'score', 'domain', 'id', 'title', 'author', 'ups', 'downs', 'num_comments', 'permalink', 'selftext', 'link_flair_text', 'over_18', 'thumbnail', 'subreddit_id', 'edited', 'link_flair_css_class', 'author_flair_css_class', 'is_self', 'name', 'url', 'distinguished']


In [8]:
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=300, centers=4, random_state=42)

In [13]:
print(df.head())

    created_utc  score       domain      id  \
0  1.362692e+09    911  i.imgur.com  19vc1i   
1  1.360106e+09    580    imgur.com  17yoix   
2  1.354915e+09    483    imgur.com  14gpws   
3  1.369706e+09    466  i.imgur.com  1f69k2   
4  1.367958e+09    461  i.imgur.com  1dvts8   

                                               title       author   ups  \
0     How I feel looking at the front page of reddit    chiachia2  1013   
1  Me as Starbuck, with President Roslin and Admi...  _galacticat   628   
2            Just Another Reason Why BSG Makes Sense    Pharazlyg   548   
3  A homemade bleached shirt I made: Launch alert...   Ferenginar   514   
4  The owner saw me taking a picture, said "Watch...       dishie   507   

   downs  num_comments                                          permalink  \
0    102            34  http://www.reddit.com/r/BSG/comments/19vc1i/ho...   
1     48            49  http://www.reddit.com/r/BSG/comments/17yoix/me...   
2     65            60  http://www.

In [15]:
print(df.columns)
print(df.head())

Index(['created_utc', 'score', 'domain', 'id', 'title', 'author', 'ups',
       'downs', 'num_comments', 'permalink', 'selftext', 'link_flair_text',
       'over_18', 'thumbnail', 'subreddit_id', 'edited',
       'link_flair_css_class', 'author_flair_css_class', 'is_self', 'name',
       'url', 'distinguished'],
      dtype='str')
    created_utc  score       domain      id  \
0  1.362692e+09    911  i.imgur.com  19vc1i   
1  1.360106e+09    580    imgur.com  17yoix   
2  1.354915e+09    483    imgur.com  14gpws   
3  1.369706e+09    466  i.imgur.com  1f69k2   
4  1.367958e+09    461  i.imgur.com  1dvts8   

                                               title       author   ups  \
0     How I feel looking at the front page of reddit    chiachia2  1013   
1  Me as Starbuck, with President Roslin and Admi...  _galacticat   628   
2            Just Another Reason Why BSG Makes Sense    Pharazlyg   548   
3  A homemade bleached shirt I made: Launch alert...   Ferenginar   514   
4  The ow

In [20]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

# Define your data
documents = [
    "I love machine learning",
    "Football is a great sport",
    "Python is used in data science",
    "I enjoy watching football",
    "AI and machine learning are powerful"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

k = 2
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(X)

terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"\nCluster {i}:")
    center = kmeans.cluster_centers_[i]
    top_words = [terms[ind] for ind in center.argsort()[-10:]]
    print(top_words)


Cluster 0:
['science', 'python', 'watching', 'in', 'great', 'enjoy', 'data', 'used', 'football', 'is']

Cluster 1:
['enjoy', 'data', 'great', 'love', 'powerful', 'are', 'and', 'ai', 'learning', 'machine']


In [21]:
df.to_csv("BSG_clustered.csv", index=False)